<a href="https://colab.research.google.com/github/thejosephineterry/antitrust-case-studies/blob/main/antitrust_case_studies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Antitrust case studies: cartel damages and merger review

Two small competition-economics case studies in Python, on **simulated or illustrative data**.
They follow the logic of work done for competition authorities and courts: build the *but-for* world (what would have happened without the conduct), measure the gap, and test how robust the answer is.


## Part 0. Supporting models: Cournot competition and collusion

*Theory:* in a Cournot market each firm picks a quantity, taking rivals' quantities as given. With P = 100 − Q and cost 20, four firms give price 36; three firms give price 40. A merger raises price, and collusion is easier to sustain when firms are few (critical discount factor).

In [ ]:
"""Cournot competition with linear demand P = a - b*Q and constant marginal cost.

Idea: each firm chooses a quantity, taking rivals' quantities as given.
A Nash equilibrium is a set of quantities where no firm gains by changing its own.
"""

def cournot_symmetric(n, a=100.0, b=1.0, c=20.0):
    q = (a - c) / (b * (n + 1))          # equilibrium output per firm
    Q = n * q                            # total output
    P = a - b * Q                        # market price
    return {"n": n, "q": q, "Q": Q, "P": P, "profit": (P - c) * q}

def hhi_equal(n):
    """Herfindahl-Hirschman Index for n equal firms, on the 0-10,000 scale."""
    return 10000.0 / n

def elasticity(P, Q, b=1.0):
    """Absolute price elasticity of demand at (P, Q) for linear demand."""
    return (1.0 / b) * P / Q

def lerner(P, c):
    """Lerner index: the share of the price that is markup over marginal cost."""
    return (P - c) / P

def cournot_asymmetric(costs, a=100.0, b=1.0):
    """Cournot equilibrium when firms have different marginal costs.
    Assumes every firm produces a positive quantity."""
    n = len(costs)
    total = sum(costs)
    Q = (n * a - total) / (b * (n + 1))
    qs = [(a - n * ci + (total - ci)) / (b * (n + 1)) for ci in costs]
    P = a - b * Q
    profits = [(P - ci) * qi for ci, qi in zip(costs, qs)]
    return {"P": P, "Q": Q, "q": qs, "profit": profits}

def merger_outcome(cm, c=20.0, a=100.0, b=1.0):
    """Two of four firms merge into one with marginal cost cm; two rivals keep cost c."""
    res = cournot_asymmetric([cm, c, c], a, b)
    return {"P": res["P"], "Q": res["Q"], "merged_profit": res["profit"][0]}

def _bisect(f, lo, hi, iters=80):
    """Find x in [lo, hi] where f(x) changes sign. Assumes f(lo) and f(hi) differ in sign."""
    flo = f(lo)
    for _ in range(iters):
        mid = (lo + hi) / 2
        if (f(mid) > 0) == (flo > 0):
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

def breakeven_cost(c=20.0, a=100.0, b=1.0):
    """Highest merged-firm cost at which the merger is privately profitable
    (merged profit equals the two firms' combined pre-merger profit)."""
    pre = 2 * cournot_symmetric(4, a, b, c)["profit"]
    return _bisect(lambda cm: merger_outcome(cm, c, a, b)["merged_profit"] - pre, 0.0, c)

def price_neutral_cost(c=20.0, a=100.0, b=1.0):
    """Merged-firm cost at which the post-merger price equals the pre-merger price."""
    p0 = cournot_symmetric(4, a, b, c)["P"]
    return _bisect(lambda cm: merger_outcome(cm, c, a, b)["P"] - p0, 0.0, c)


In [ ]:
"""Repeated games and collusion, plus a simple leniency game."""
from itertools import product

def delta_star(n, a=100.0, b=1.0, c=20.0):
    """Critical discount factor for grim-trigger collusion among n Cournot firms.
    Collusion holds if delta >= (pi_dev - pi_coll) / (pi_dev - pi_nash)."""
    Qm = (a - c) / (2 * b)                         # monopoly output
    pi_coll = (a - b * Qm - c) * Qm / n            # each firm's share of monopoly profit
    Q_others = Qm * (n - 1) / n                    # rivals keep to the collusive output
    q_dev = (a - c - b * Q_others) / (2 * b)       # best deviation
    pi_dev = (a - b * (Q_others + q_dev) - c) * q_dev
    pi_nash = ((a - c) / (b * (n + 1))) ** 2 * b   # punishment: Cournot profit
    return (pi_dev - pi_coll) / (pi_dev - pi_nash)

def delta_star_2x2(coll, dev, nash):
    """Same formula for a 2x2 payoff game."""
    return (dev - coll) / (dev - nash)

def pure_nash(payoffs):
    """Pure-strategy Nash equilibria of a 2-player game.
    payoffs[(row, col)] = (payoff_row_player, payoff_col_player)."""
    rows = sorted({r for r, _ in payoffs})
    cols = sorted({c for _, c in payoffs})
    eq = []
    for r, c in product(rows, cols):
        best_r = all(payoffs[(r, c)][0] >= payoffs[(r2, c)][0] for r2 in rows)
        best_c = all(payoffs[(r, c)][1] >= payoffs[(r, c2)][1] for c2 in cols)
        if best_r and best_c:
            eq.append((r, c))
    return eq

# Leniency game: after an investigation starts, each bank can Confess or stay Silent.
LENIENCY = {
    ("Confess", "Confess"): (-8, -8),
    ("Confess", "Silent"): (0, -20),
    ("Silent", "Confess"): (-20, 0),
    ("Silent", "Silent"): (-16, -16),
}


## Case 1. Cartel overcharge and damages

*Story:* a long-running cartel raises prices. A court asks: how much higher were prices *because of* the cartel?  
*Method:* compare a naive estimate, a regression with cost and demand controls (robust standard errors), and a counterfactual forecast. Then convert the overcharge into damages with a **pass-on** effect and a **quantity** effect.  
*Key lesson:* leaving out the cost control biases the answer, which is why robustness checks decide whether evidence survives challenge.

In [ ]:
"""Case study 1: estimating a cartel overcharge and the damages a direct purchaser can claim.

Modelled on the structure of a trucks-style cartel: a long infringement period, cost and
regulation shocks that move prices for reasons unrelated to the cartel, and three questions
a court will ask: (1) what would prices have been 'but for' the cartel, (2) how robust is
that estimate, (3) how much of the overcharge did the buyer pass on?
All data here are simulated, so the true overcharge is known and the methods can be tested.
"""
import numpy as np

def simulate_market(seed=7, n_months=300, cartel_start=60, cartel_end=228, true_log_overcharge=0.06):
    """Monthly log price = f(cost, demand) + cartel effect + autocorrelated noise."""
    rng = np.random.default_rng(seed)
    t = np.arange(n_months)
    cartel = ((t >= cartel_start) & (t < cartel_end)).astype(float)
    regulation = (t >= 150).astype(float)                       # a new emissions standard raises costs
    # Costs trend up faster during the cartel years: the confounder that biases naive comparisons.
    ln_cost = 4.6 + 0.0015 * t + 0.0020 * cartel * (t - cartel_start) + 0.05 * regulation + rng.normal(0, 0.02, n_months)
    ln_demand = 4.6 + 0.0010 * t + 0.12 * np.sin(2 * np.pi * t / 12) + rng.normal(0, 0.03, n_months)
    eps = np.zeros(n_months)
    for i in range(1, n_months):
        eps[i] = 0.5 * eps[i - 1] + rng.normal(0, 0.02)          # persistent shocks
    ln_price = 1.0 + 0.55 * ln_cost + 0.25 * ln_demand + true_log_overcharge * cartel + eps
    return {"t": t, "cartel": cartel, "ln_cost": ln_cost, "ln_demand": ln_demand, "ln_price": ln_price}

def ols(y, X):
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return beta, y - X @ beta

def newey_west_se(X, resid, lags):
    """Heteroskedasticity-and-autocorrelation-robust standard errors."""
    xtx_inv = np.linalg.inv(X.T @ X)
    u = X * resid[:, None]
    S = u.T @ u
    for lag in range(1, lags + 1):
        w = 1 - lag / (lags + 1)
        G = u[lag:].T @ u[:-lag]
        S += w * (G + G.T)
    return np.sqrt(np.diag(xtx_inv @ S @ xtx_inv))

def estimate_overcharge(data, lags=12):
    d = data
    n = len(d["t"])
    one = np.ones(n)
    out = {}
    # (A) Naive before/during comparison with no controls.
    out["naive"] = d["ln_price"][d["cartel"] == 1].mean() - d["ln_price"][d["cartel"] == 0].mean()
    # (B) Dummy-variable regression with cost and demand controls.
    X = np.column_stack([one, d["ln_cost"], d["ln_demand"], d["cartel"]])
    beta, resid = ols(d["ln_price"], X)
    se = newey_west_se(X, resid, lags)
    out["dummy"], out["dummy_se"] = beta[-1], se[-1]
    # (C) Forecast approach: fit on non-cartel months, predict the cartel months, average the gap.
    mask = d["cartel"] == 0
    Xf = np.column_stack([one, d["ln_cost"], d["ln_demand"]])
    bf, _ = ols(d["ln_price"][mask], Xf[mask])
    out["forecast"] = (d["ln_price"][~mask] - Xf[~mask] @ bf).mean()
    # (D) Robustness: omit the cost control.
    Xo = np.column_stack([one, d["ln_demand"], d["cartel"]])
    out["omit_cost"] = ols(d["ln_price"], Xo)[0][-1]
    return out

def direct_purchaser_damages(price_cartel, units, overcharge_pct, pass_on, elasticity, downstream_price, margin):
    """Overcharge, price effect (pass-on) and quantity effect for a direct purchaser."""
    but_for = price_cartel / (1 + overcharge_pct)
    oc_unit = price_cartel - but_for
    overcharge_total = oc_unit * units
    pass_on_benefit = pass_on * oc_unit * units                      # price effect
    pct_rise = pass_on * oc_unit / downstream_price
    lost_units = elasticity * pct_rise * units                        # quantity effect
    lost_profit = lost_units * margin
    return {
        "overcharge": overcharge_total,
        "pass_on_benefit": pass_on_benefit,
        "quantity_effect": lost_profit,
        "net_damages": overcharge_total - pass_on_benefit + lost_profit,
    }


## Case 2. Merger review across local markets

*Story:* two retailers plan to merge. Which local markets raise concerns, and would selling stores fix it?  
*Method:* overlap, HHI before and after, upward pricing pressure (GUPPI) from share-based diversion, then a divestiture test comparing a sale to a rival with a sale to a new entrant.

In [ ]:
"""Case study 2: merger review across local markets (concentration screen, unilateral effects, remedy).

Two retailers, A and B, plan to merge. Each local market has different rivals. The questions an
authority and the parties' economists ask: where do the firms overlap, where does the merger
raise unilateral-effects concerns, and would divesting stores in the problem markets fix it?
Market shares are illustrative, not real data.
"""

# Shares in percent: (A, B, [rivals]).
MARKETS = {
    "M1": (30, 25, [20, 15, 10]),
    "M2": (40, 5,  [30, 25]),
    "M3": (20, 20, [20, 20, 20]),
    "M4": (0, 35,  [40, 25]),
    "M5": (45, 30, [25]),
    "M6": (15, 10, [25, 25, 25]),
    "M7": (35, 0,  [35, 30]),
    "M8": (25, 25, [50]),
}

def hhi(shares):
    return sum(s * s for s in shares)

def analyse_market(a, b, rivals, margin=0.35, hhi_delta=200, hhi_post=2000, guppi_flag=0.05):
    pre = hhi([a, b] + rivals)
    delta = 2 * a * b
    post = pre + delta
    d_ab = b / (100 - a) if a < 100 else 0.0        # diversion A->B, assuming proportional to share
    d_ba = a / (100 - b) if b < 100 else 0.0
    guppi_a, guppi_b = d_ab * margin, d_ba * margin  # equal prices assumed
    overlap = a > 0 and b > 0
    concern = overlap and ((delta >= hhi_delta and post >= hhi_post) or max(guppi_a, guppi_b) >= guppi_flag)
    return {"pre": pre, "post": post, "delta": delta, "guppi_a": guppi_a, "guppi_b": guppi_b,
            "overlap": overlap, "concern": concern}

def remedy(a, b, rivals):
    """Divest the smaller party's local store to the smallest existing rival.
    Returns the HHI change versus the pre-merger market. Selling to a NEW independent entrant
    would leave the change at zero, so compare the two buyers."""
    if not rivals:
        return None
    smaller = min(a, b)
    larger = max(a, b)
    rivals = sorted(rivals)
    new_rivals = [rivals[0] + smaller] + rivals[1:]
    after = hhi([larger] + new_rivals)
    before = hhi([a, b] + rivals)
    return after - before

def review(markets=MARKETS):
    rows = {}
    for name, (a, b, rv) in markets.items():
        r = analyse_market(a, b, rv)
        r["remedy_delta"] = remedy(a, b, rv) if r["overlap"] else 0
        rows[name] = r
    return rows


## Checking the results

In [ ]:
import numpy as np
"""Runs both antitrust case studies plus the supporting models, prints results and checks them."""

print("== Case 1: cartel overcharge (simulated 25-year market, 14-year cartel, true overcharge 6.0% in logs) ==")
data = simulate_market()
est = estimate_overcharge(data)
print(f"Naive before/during comparison: {est['naive']:.1%}  (biased: cost rose faster during the cartel)")
print(f"Dummy regression with controls: {est['dummy']:.1%}  (HAC s.e. {est['dummy_se']:.1%})")
print(f"Forecast (counterfactual) approach: {est['forecast']:.1%}")
print(f"Omitting the cost control: {est['omit_cost']:.1%}")
o_pct = np.exp(est["dummy"]) - 1
dmg = direct_purchaser_damages(price_cartel=100_000, units=1_000, overcharge_pct=o_pct, pass_on=0.6,
                               elasticity=1.2, downstream_price=130_000, margin=15_000)
print(f"Overcharge as % of but-for price: {o_pct:.1%}")
print({k: round(float(v)) for k, v in dmg.items()})
print("Sensitivity of net damages to pass-on rate:",
      {p: round(float(direct_purchaser_damages(100_000, 1_000, o_pct, p, 1.2, 130_000, 15_000)['net_damages']) / 1e6, 2) for p in (0.0, 0.3, 0.6, 0.9)}, "(EUR m)")
assert abs(est["dummy"] - 0.06) < 0.02 and est["naive"] > est["dummy"] + 0.05

print("\n== Case 2: merger review across 8 local markets ==")
res = review()
print(f"{'Mkt':4}{'Overlap':8}{'PreHHI':8}{'PostHHI':9}{'Delta':7}{'GUPPI_A':9}{'GUPPI_B':9}{'Concern':9}{'dHHI if sold to rival':22}")
for k, r in res.items():
    print(f"{k:4}{str(r['overlap']):8}{r['pre']:<8}{r['post']:<9}{r['delta']:<7}{r['guppi_a']:<9.3f}{r['guppi_b']:<9.3f}{str(r['concern']):9}{r['remedy_delta']:<12}")
n_overlap = sum(r["overlap"] for r in res.values()); n_flag = sum(r["concern"] for r in res.values())
print(f"Overlap markets: {n_overlap} of {len(res)}; flagged: {n_flag}")
worst = max(res.items(), key=lambda kv: kv[1]["delta"])
print(f"Largest HHI increase: {worst[0]} (+{worst[1]['delta']}); a sale to a new entrant removes it, a sale to the smallest rival leaves +{worst[1]['remedy_delta']}")
still = [k for k, r in res.items() if r["overlap"] and r["remedy_delta"] >= 200]
print(f"Markets still above the 200-point screen after selling to the smallest rival: {len(still)} of {n_overlap} -> {still}")
print(f"Markets above the screen after selling to a new entrant: 0 of {n_overlap}")
assert n_flag == 6 and len(still) == 6

print("\n== Supporting national-level model (Cournot 4-to-3) ==")
print(f"Price {cournot_symmetric(4)['P']:.0f} -> {cournot_symmetric(3)['P']:.0f}; merged firm needs {breakeven_cost():.1f} cost for break-even, {price_neutral_cost():.1f} to hold price")
print(f"Critical discount factor: 3 firms {delta_star(3):.3f}, 5 firms {delta_star(5):.3f}")
print("\nAll checks passed.")


== Case 1: cartel overcharge (simulated 25-year market, 14-year cartel, true overcharge 6.0% in logs) ==
Naive before/during comparison: 13.3%  (biased: cost rose faster during the cartel)
Dummy regression with controls: 6.2%  (HAC s.e. 0.6%)
Forecast (counterfactual) approach: 5.9%
Omitting the cost control: 14.7%
Overcharge as % of but-for price: 6.4%
{'overcharge': 5971326, 'pass_on_benefit': 3582796, 'quantity_effect': 496079, 'net_damages': 2884610}
Sensitivity of net damages to pass-on rate: {0.0: 5.97, 0.3: 4.43, 0.6: 2.88, 0.9: 1.34} (EUR m)

== Case 2: merger review across 8 local markets ==
Mkt Overlap PreHHI  PostHHI  Delta  GUPPI_A  GUPPI_B  Concern  dHHI if sold to rival 
M1  True    2250    3750     1500   0.125    0.140    True     500         
M2  True    3150    3550     400    0.029    0.147    True     250         
M3  True    2000    2800     800    0.087    0.087    True     800         
M4  False   3450    3450     0      0.122    0.000    False    0           
M5

## Limits
Simulated data; identical products and linear demand in the supporting models; diversion assumed proportional to share; screen thresholds are illustrative parameters, not any authority's rules. A real case needs transaction data, cost data, diversion evidence and a court-ready robustness analysis.